## 1. Составление популярности айтемов

In [ ]:
from collections import defaultdict
import numpy as np

def compute_item_popularity(interactions):
    """
    interactions: list of (user_id, item_id, weight) tuples
                  weight — количество взаимодействий
    returns: dict {item_id: popularity_score}
    """
    item_popularity = defaultdict(float)
    for user, item, weight in interactions:
        item_popularity[item] += weight
    return item_popularity

## 2. Выбор top-K популярных айтемов

In [ ]:
def get_top_popular_items(item_popularity, top_k):
    """
    item_popularity: dict {item_id: score}
    top_k: сколько взять самых популярных
    returns: list of top_k item_ids
    """
    sorted_items = sorted(item_popularity.keys(), key=lambda x: -item_popularity[x])
    return sorted_items[:top_k]

## 3. Выбор наиболее разнообразных и популярных (Maximum Volume / greedy)

In [ ]:
def select_diverse_popular_items(top_items, item_embs, item_popularity, final_k, alpha=0.7):
    """
    top_items: list of item_ids (от самых популярных)
    item_embs: dict {item_id: embedding vector}
    item_popularity: dict {item_id: score}
    final_k: сколько выбрать после diversification
    alpha: баланс популярность vs разнообразие (0-1)
    returns: list of final_k item_ids
    """
    
    # Нормализуем embeddings
    norm_embs = {i: e/np.linalg.norm(e) for i, e in item_embs.items() if i in top_items}
    
    selected = []
    remaining = list(top_items)
    
    for _ in range(final_k):
        best_item = None
        best_score = -np.inf
        
        for item in remaining:
            e = norm_embs[item]
            # diversity = минимальная схожесть с уже выбранными
            if not selected:
                diversity = 0
            else:
                diversity = -max(np.dot(e, norm_embs[s]) for s in selected)
            
            # популярность с лог-нормализацией
            pop = np.log(1 + item_popularity[item])
            
            # комбинированный скор
            score = alpha * pop + (1 - alpha) * diversity
            
            if score > best_score:
                best_score = score
                best_item = item
                
        selected.append(best_item)
        remaining.remove(best_item)
    
    return selected

## 4. Функция подготовки данных для Factorization Machine

In [ ]:
import pandas as pd
from itertools import product

def prepare_fm_training_data(cold_users, candidate_items, user_features, item_features, interactions):
    """
    cold_users: list of user_id
    candidate_items: list of item_id (top K popular/diverse)
    user_features: dict {user_id: np.array}
    item_features: dict {item_id: np.array}
    interactions: set of (user_id, item_id) positive interactions
    """
    X, y = [], []
    
    for u, i in product(cold_users, candidate_items):
        uf = user_features[u]
        itf = item_features[i]
        features = np.concatenate([uf, itf])
        X.append(features)
        # target: 1 если взаимодействие есть, иначе 0
        y.append(1 if (u, i) in interactions else 0)
        
    return np.array(X), np.array(y)

## 5. Обучение модели CatBoost для cold users

In [ ]:
from catboost import CatBoostClassifier

def train_catboost(X, y):
    """
    Обучаем CatBoost для ранжирования cold users
    """
    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function='Logloss',  # можно заменить на 'PairLogit' если есть пары
        verbose=100
    )
    model.fit(X, y)
    return model

# 6. Пример использования

In [ ]:
# 1. Считаем популярность
item_pop = compute_item_popularity(interactions)

# 2. Берем топ популярных (Задаем параметр top_k)
top_pop_items = get_top_popular_items(item_pop, top_k=500)

# 3. Выбираем топ с балансом популярность + разнообразие
#   Задаем параметр: final_k - сколько хотим получить в конце
#   Задаем параметр: alpha - насколько важна популярность и разнообразность
#  (чем больше alpha, тем важнее популярность)
final_items = select_diverse_popular_items(
    top_pop_items, item_embs, item_pop, final_k=50, alpha=0.7
)